<a href="https://colab.research.google.com/github/Iditc/log-anomaly-detection/blob/main/notebooks/01_parsing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

BASE = Path('/content/drive/MyDrive/log-anomaly-detection')
DATA_RAW = BASE / 'data' / 'raw'
DATA_PROC = BASE / 'data' / 'processed'
RESULTS = BASE / 'results'

# Create the folders if they do not exist
DATA_RAW.mkdir(parents=True, exist_ok=True)
DATA_PROC.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

print("Folders created:")
print(DATA_RAW)
print(DATA_PROC)
print(RESULTS)


תיקיות נוצרו:
/content/drive/MyDrive/log-anomaly-detection/data/raw
/content/drive/MyDrive/log-anomaly-detection/data/processed
/content/drive/MyDrive/log-anomaly-detection/results


In [4]:
import urllib.request

# Download HDFS log dataset from Loghub GitHub
url = "https://raw.githubusercontent.com/logpai/loghub/master/HDFS/HDFS_2k.log"
dest = DATA_RAW / "HDFS_2k.log"

print("Downloading HDFS sample...")
urllib.request.urlretrieve(url, dest)
print("Done. File saved to:", dest)

Done. File saved to: /content/drive/MyDrive/log-anomaly-detection/data/raw/HDFS_2k.log


In [6]:
# Read and display first 5 lines of the log file
with open(DATA_RAW / "HDFS_2k.log", "r") as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        print(line.strip())


081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049064139660 terminating
081109 203807 222 INFO dfs.DataNode$PacketResponder: PacketResponder 0 for block blk_-6952295868487656571 terminating
081109 204005 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.73.220:50010 is added to blk_7128370237687728475 size 67108864
081109 204015 308 INFO dfs.DataNode$PacketResponder: PacketResponder 2 for block blk_8229193803249955061 terminating
081109 204106 329 INFO dfs.DataNode$PacketResponder: PacketResponder 2 for block blk_-6670958622368987959 terminating
081109 204132 26 INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.43.115:50010 is added to blk_3050920587428079149 size 67108864
081109 204324 34 INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.203.80:50010 is added to blk_7888946331804732825 size 67108864
081109 204453 34 INFO dfs.FSNamesystem: BLOCK*

In [8]:
!pip install drain3 -q

  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyiceberg 0.11.1 requires cachetools<7.0,>=5.5, but you have cachetools 4.2.1 which is incompatible.


In [9]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

# Configure Drain3 parser
config = TemplateMinerConfig()
config.drain_depth = 4
config.drain_sim_th = 0.5

miner = TemplateMiner(config=config)
print("Parser ready.")

Parser ready.


In [10]:
# Parse all log lines and extract templates
with open(DATA_RAW / "HDFS_2k.log", "r") as f:
    for line in f:
        miner.add_log_message(line.strip())

print(f"Total templates found: {len(miner.drain.clusters)}")

Total templates found: 21


In [11]:
# Print all discovered log templates
for cluster in miner.drain.clusters:
    print(f"ID {cluster.cluster_id:2d} | Count: {cluster.size:4d} | Template: {cluster.get_template()}")

ID  1 | Count:  311 | Template: <*> <*> <*> INFO dfs.DataNode$PacketResponder: PacketResponder <*> for block <*> terminating
ID  2 | Count:  314 | Template: <*> <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: <*> is added to <*> size <*>
ID  3 | Count:  292 | Template: <*> <*> <*> INFO dfs.DataNode$PacketResponder: Received block <*> of size <*> from <*>
ID  4 | Count:  292 | Template: <*> <*> <*> INFO dfs.DataNode$DataXceiver: Receiving block <*> src: <*> dest: <*>
ID  5 | Count:    8 | Template: 081109 <*> <*> INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: <*> <*>
ID  6 | Count:   20 | Template: <*> <*> 13 INFO dfs.DataBlockScanner: Verification succeeded for <*>
ID  7 | Count:  263 | Template: <*> <*> <*> INFO dfs.FSDataset: Deleting block <*> file <*>
ID  8 | Count:    9 | Template: 081109 <*> <*> INFO dfs.DataNode$DataXceiver: <*> Served block <*> to <*>
ID  9 | Count:   80 | Template: <*> <*> <*> WARN dfs.DataNode$DataXceiver: <*> excep

In [12]:
import pandas as pd

# Assign template ID to each log line
records = []
with open(DATA_RAW / "HDFS_2k.log", "r") as f:
    for line in f:
        line = line.strip()
        result = miner.add_log_message(line)
        records.append({
            "log_line": line,
            "template_id": result["cluster_id"],
            "template": result["template_mined"]
        })

df = pd.DataFrame(records)
df.to_parquet(DATA_PROC / "hdfs_parsed.parquet", index=False)
print(f"Saved {len(df)} rows to parquet.")
print(df.head())

Saved 2000 rows to parquet.
                                            log_line  template_id  \
0  081109 203615 148 INFO dfs.DataNode$PacketResp...            1   
1  081109 203807 222 INFO dfs.DataNode$PacketResp...            1   
2  081109 204005 35 INFO dfs.FSNamesystem: BLOCK*...            2   
3  081109 204015 308 INFO dfs.DataNode$PacketResp...            1   
4  081109 204106 329 INFO dfs.DataNode$PacketResp...            1   

                                            template  
0  <*> <*> <*> INFO dfs.DataNode$PacketResponder:...  
1  <*> <*> <*> INFO dfs.DataNode$PacketResponder:...  
2  <*> <*> <*> INFO dfs.FSNamesystem: BLOCK* Name...  
3  <*> <*> <*> INFO dfs.DataNode$PacketResponder:...  
4  <*> <*> <*> INFO dfs.DataNode$PacketResponder:...  


In [16]:
import requests

# Check what files are available on Zenodo
response = requests.get("https://zenodo.org/api/records/8196385")
data = response.json()
for f in data["files"]:
    print(f["key"], "—", f["size"], "bytes")


Android_v1.zip — 24858007 bytes
Android_v2.zip — 444986139 bytes
Zookeeper.tar.gz — 459911 bytes
Windows.tar.gz — 1670098945 bytes
Thunderbird.tar.gz — 2016100298 bytes
SSH.tar.gz — 4596666 bytes
Spark.tar.gz — 183474743 bytes
Proxifier.tar.gz — 172346 bytes
OpenStack.tar.gz — 5394691 bytes
Mac.tar.gz — 1489372 bytes
Linux.tar.gz — 232039 bytes
HPC.zip — 2975898 bytes
HealthApp.tar.gz — 2296618 bytes
HDFS_v3_TraceBench.zip — 567419398 bytes
HDFS_v2.zip — 823700713 bytes
HDFS_v1.zip — 186645559 bytes
Hadoop.zip — 3416419 bytes
BGL.zip — 57489019 bytes
Apache.tar.gz — 262905 bytes


In [17]:
import zipfile
import io

# Download HDFS_v1.zip from Zenodo
print("Downloading HDFS_v1.zip (~186MB)...")
response = requests.get(
    "https://zenodo.org/records/8196385/files/HDFS_v1.zip",
    stream=True
)

dest = DATA_RAW / "HDFS_v1.zip"
with open(dest, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print("Download complete. Extracting...")
with zipfile.ZipFile(dest, "r") as z:
    z.extractall(DATA_RAW)
    print("Files extracted:", z.namelist())

Download complete. Extracting...
Files extracted: ['HDFS.log', 'preprocessed/', 'preprocessed/anomaly_label.csv', 'preprocessed/Event_occurrence_matrix.csv', 'preprocessed/Event_traces.csv', 'preprocessed/HDFS.log_templates.csv', 'preprocessed/HDFS.npz', 'README.md']


In [18]:
# Check log file size and label distribution
import os

log_path = DATA_RAW / "HDFS.log"
label_path = DATA_RAW / "preprocessed" / "anomaly_label.csv"

# Log file
print(f"Log file size: {log_path.stat().st_size / 1e6:.1f} MB")

# Labels
labels = pd.read_csv(label_path)
print(f"\nLabels shape: {labels.shape}")
print(labels.head())
print(f"\nLabel distribution:\n{labels['Label'].value_counts()}")

Log file size: 1578.0 MB

Labels shape: (575061, 2)
                    BlockId    Label
0  blk_-1608999687919862906   Normal
1   blk_7503483334202473044   Normal
2  blk_-3544583377289625738  Anomaly
3  blk_-9073992586687739851   Normal
4   blk_7854771516489510256   Normal

Label distribution:
Label
Normal     558223
Anomaly     16838
Name: count, dtype: int64


In [19]:
import re

# Extract block ID from each log line and join with labels
print("Reading log file...")
records = []
pattern = re.compile(r"(blk_-?\d+)")

with open(log_path, "r") as f:
    for line in f:
        match = pattern.search(line)
        block_id = match.group(1) if match else None
        records.append({
            "log_line": line.strip(),
            "block_id": block_id
        })

df_full = pd.DataFrame(records)
print(f"Total lines: {len(df_full):,}")
print(f"Lines with block ID: {df_full['block_id'].notna().sum():,}")

Reading log file...
Total lines: 11,175,629
Lines with block ID: 11,175,629


In [23]:
# Rename column to match and join
labels_renamed = labels.rename(columns={"BlockId": "block_id"})
df_full = df_full.merge(labels_renamed, on="block_id", how="left")

print(f"Label distribution in log lines:")
print(df_full["Label"].value_counts())
print(f"\nLines without label: {df_full['Label'].isna().sum():,}")


Label distribution in log lines:
Label
Normal     10887379
Anomaly      288250
Name: count, dtype: int64

Lines without label: 0


In [24]:
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig

# Configure Drain3
config = TemplateMinerConfig()
config.drain_depth = 4
config.drain_sim_th = 0.5

miner = TemplateMiner(config=config)

# Parse all log lines
print("Parsing log lines... (this will take ~10-15 minutes)")
template_ids = []
for i, line in enumerate(df_full["log_line"]):
    result = miner.add_log_message(line)
    template_ids.append(result["cluster_id"])
    if i % 1_000_000 == 0:
        print(f"  Processed {i:,} lines...")

df_full["template_id"] = template_ids
print(f"Done! Total templates found: {len(miner.drain.clusters)}")


Parsing log lines... (this will take ~10-15 minutes)
  Processed 0 lines...
  Processed 1,000,000 lines...
  Processed 2,000,000 lines...
  Processed 3,000,000 lines...
  Processed 4,000,000 lines...
  Processed 5,000,000 lines...
  Processed 6,000,000 lines...
  Processed 7,000,000 lines...
  Processed 8,000,000 lines...
  Processed 9,000,000 lines...
  Processed 10,000,000 lines...
  Processed 11,000,000 lines...
Done! Total templates found: 55


- `log_line` — השורה המקורית
- `block_id` — מזהה ה-block
- `Label` — Normal / Anomaly
- `template_id` — מספר ה-template מ-Drain3

In [25]:
# Save full parsed dataset to parquet
print("Saving to parquet...")
df_full.to_parquet(DATA_PROC / "hdfs_full_parsed.parquet", index=False)
print(f"Saved {len(df_full):,} rows.")
print(df_full.head())


Saving to parquet...
Saved 11,175,629 rows.
                                            log_line  \
0  081109 203518 143 INFO dfs.DataNode$DataXceive...   
1  081109 203518 35 INFO dfs.FSNamesystem: BLOCK*...   
2  081109 203519 143 INFO dfs.DataNode$DataXceive...   
3  081109 203519 145 INFO dfs.DataNode$DataXceive...   
4  081109 203519 145 INFO dfs.DataNode$PacketResp...   

                   block_id   Label  template_id  
0  blk_-1608999687919862906  Normal            1  
1  blk_-1608999687919862906  Normal            2  
2  blk_-1608999687919862906  Normal            1  
3  blk_-1608999687919862906  Normal            1  
4  blk_-1608999687919862906  Normal            3  
